In [81]:
import pandas as pd
import numpy as np

In [82]:
trainPath = "Data/train.csv"
testPath = "Data/test.csv"

In [83]:
train = pd.read_csv(trainPath)
test = pd.read_csv(testPath)

In [84]:
print("train: ", train.shape)
print("test: ", test.shape)

train:  (7709, 25)
test:  (2508, 24)


In [85]:
TARGET = "overqualified"
ID_COL = "id"

# seperating our data into an input set and an output set
X_train = train.drop(columns = [TARGET])
y_train = train[TARGET].values

X_test = test.copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (7709, 24)
y_train shape: (7709,)
X_test shape : (2508, 24)


In [86]:
MISSING_CODES = [6, 9, 99]

def replace_missing_codes(df):
    df = df.copy()
    for col in df.columns:
        if col == "id":
            continue
        # only try replacing in numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].replace(MISSING_CODES, np.nan)
    return df

X_train_clean = replace_missing_codes(X_train)
X_test_clean  = replace_missing_codes(X_test)

print("Missing % after replacement (top 10 columns):")
print(X_train_clean.isna().mean().sort_values(ascending=False).head(10))


Missing % after replacement (top 10 columns):
PGM_P036    0.831236
DBTOTGRD    0.793618
SCHOLARP    0.731094
PREVLEVP    0.557530
BEF_160     0.525619
PGMCIPAP    0.293553
PAR2GRD     0.241925
PAR1GRD     0.222597
PGM_280B    0.182644
PGM_280C    0.182125
dtype: float64


In [87]:
categorical_cols = [
    "CERTLEVP",
    "PGMCIPAP",
    "PGM_P034",
    "PGM_P036",
    "PGM_280A",
    "PGM_280B",
    "PGM_280C",
    "PGM_280F",
    "PGM_P401",
    "STULOANS",
    "GENDER2",
    "CTZSHIPP",
    "VISBMINP",
    "DDIS_FL",
    "PAR1GRD",
    "PAR2GRD",
    "BEF_P140",
]

numeric_cols = [
    "BEF_160",
]

In [88]:
print("Missing categorical columns:",
      [c for c in categorical_cols if c not in X_train_clean.columns])

print("Missing numeric columns:",
      [c for c in numeric_cols if c not in X_train_clean.columns])


Missing categorical columns: []
Missing numeric columns: []


In [89]:
X_train_filled = X_train_clean.copy()
X_test_filled  = X_test_clean.copy()

# 1) Numeric: fill with median (learn median from train only)
for col in numeric_cols:
    med = X_train_filled[col].median()
    X_train_filled[col] = X_train_filled[col].fillna(med)
    X_test_filled[col]  = X_test_filled[col].fillna(med)

# 2) Categorical: fill with "Missing"
for col in categorical_cols:
    X_train_filled[col] = X_train_filled[col].fillna("Missing").astype(str)
    X_test_filled[col]  = X_test_filled[col].fillna("Missing").astype(str)

# quick check
print("Total missing in train:", X_train_filled.isna().sum().sum())
print("Total missing in test :", X_test_filled.isna().sum().sum())



Total missing in train: 17570
Total missing in test : 5691


In [90]:
ordinal_cols = [
    "HLOSGRDP",
    "PREVLEVP",
    "DBTOTGRD",
    "SCHOLARP",
    "GRADAGEP",
]

for col in ordinal_cols:
    X_train_filled[col] = X_train_filled[col].fillna("Missing").astype(str)
    X_test_filled[col]  = X_test_filled[col].fillna("Missing").astype(str)

# check again
print("Total missing in train:", X_train_filled.isna().sum().sum())
print("Total missing in test :", X_test_filled.isna().sum().sum())


Total missing in train: 0
Total missing in test : 0


In [91]:
encode_cols = categorical_cols + ordinal_cols

X_train_encoded = pd.get_dummies(
    X_train_filled,
    columns=encode_cols,
    drop_first=True
)

X_test_encoded = pd.get_dummies(
    X_test_filled,
    columns=encode_cols,
    drop_first=True
)

print("Before align:")
print("train:", X_train_encoded.shape)
print("test :", X_test_encoded.shape)


Before align:
train: (7709, 114)
test : (2508, 84)


In [92]:
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print("After align:")
print("train:", X_train_encoded.shape)
print("test :", X_test_encoded.shape)


After align:
train: (7709, 114)
test : (2508, 114)


In [93]:
# ========== FEATURE ENGINEERING ==========
print("Creating new features...")

# Helper function to convert safely
def safe_float(series):
    return pd.to_numeric(series, errors='coerce').fillna(0)

# For TRAIN
X_train_encoded['business_A'] = (X_train_filled['PGM_280A'] == '1').astype(int)
X_train_encoded['business_B'] = (X_train_filled['PGM_280B'] == '1').astype(int)
X_train_encoded['business_C'] = (X_train_filled['PGM_280C'] == '1').astype(int)
X_train_encoded['program_studied'] = (X_train_filled['PGMCIPAP'] == '1').astype(int)
X_train_encoded['diability'] = (X_train_filled['DDIS_FL'] == '1').astype(int)
X_train_encoded['activity'] = (X_train_filled['BEF_P140'] == '1').astype(int)


# For TEST (same features!)
X_test_encoded['business_A'] = (X_test_filled['PGM_280A'] == '1').astype(int)
X_test_encoded['business_B'] = (X_test_filled['PGM_280B'] == '1').astype(int)
X_test_encoded['business_C'] = (X_test_filled['PGM_280C'] == '1').astype(int)
X_test_encoded['program_studied'] = (X_test_filled['PGMCIPAP'] == '1').astype(int)
X_test_encoded['diability'] = (X_test_filled['DDIS_FL'] == '1').astype(int)
X_test_encoded['activity'] = (X_test_filled['BEF_P140'] == '1').astype(int)

print(f"New shape - train: {X_train_encoded.shape}, test: {X_test_encoded.shape}")

Creating new features...
New shape - train: (7709, 120), test: (2508, 120)


In [94]:
import os
os.makedirs("Data/processed", exist_ok=True)

X_train_encoded.to_csv("Data/processed/X_train_clean.csv", index=False)
pd.DataFrame(y_train, columns=['overqualified']).to_csv("Data/processed/y_train.csv", index=False)
X_test_encoded.to_csv("Data/processed/X_test_clean.csv", index=False)
print("Saved cleaned files to Data/processed/")


Saved cleaned files to Data/processed/
